# Notebook 09b: Vehicle ReID -- CityFlowV2 Fine-Tuning (384px)
**Multi-Camera Tracking System -- Kaggle Training Pipeline**

Fine-tune TransReID ViT-Base (CLIP pretrained, VeRi-776 init) on **CityFlowV2** at **384x384** --
a real multi-camera vehicle tracking dataset from the AI City Challenge 2022.

## Why CityFlowV2?
- **Real multi-camera tracking**: 46 physical cameras across city intersections
- **880 globally-consistent vehicle identities** across all cameras
- **MTMC evaluation**: IDF1, HOTA, MOTA metrics (via Stage 5)
- **Domain gap**: traffic footage != controlled studio (VeRi-776)

## Strategy
1. Extract vehicle crops from CityFlowV2 GT annotations + video frames
2. Use all available train cameras from the Kaggle-mounted CityFlowV2 dataset
3. Split into train/query/gallery (70/30 of multi-camera vehicle IDs)
4. Fine-tune TransReID from VeRi-776 pretrained weights (domain adaptation)
5. Evaluate with mAP, Rank-1 (ReID) + IDF1, HOTA (via full pipeline)

## Model
| Model | Architecture | Dim | Init | Target mAP |
|-------|-------------|-----|------|------------|
| **TransReID** | ViT-Base/16 (CLIP) + SIE + JPM | 768 | VeRi-776 | >78% |

**Runtime**: GPU T4 x2 (32GB) | **Duration**: ~3-4h (120 epochs, fine-tune, 384px, CityFlowV2)

In [ ]:
!pip install -q timm matplotlib pandas loguru gdown

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import re
import shutil
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim import Adam, SGD
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
NUM_GPUS = max(torch.cuda.device_count(), 1)
print(f"GPUs available: {NUM_GPUS}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1024**3:.1f} GB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/vehicle_reid_cityflowv2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/kaggle/working/exported_models")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nDevice: {DEVICE} | GPUs: {NUM_GPUS}")

## 1.5 Locate CityFlowV2 Kaggle Dataset

CityFlowV2 is attached as a Kaggle dataset. We locate the mounted dataset root
and use the train split directly instead of downloading from Google Drive.

In [ ]:
# -- Locate Kaggle-mounted CityFlowV2 dataset --
_CANDIDATE_MOUNTS = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]

CITYFLOW_DIR = None
for _p in _CANDIDATE_MOUNTS:
    if _p.exists():
        CITYFLOW_DIR = _p
        break

if CITYFLOW_DIR is None:
    raise FileNotFoundError(
        "CityFlowV2 dataset not found at any candidate mount:\n"
        + "\n".join(f"  - {p}" for p in _CANDIDATE_MOUNTS)
    )

print(f"CityFlowV2 dataset root: {CITYFLOW_DIR}")
print(f"Contents: {[p.name for p in CITYFLOW_DIR.iterdir()]}")

## 2. CityFlowV2 Dataset -- Crop Extraction

CityFlowV2 provides per-camera videos + MOT-format ground truth (gt.txt).
We extract vehicle bounding-box crops from video frames, then split into
train/query/gallery for ReID training.

**Using all 59 scene-camera combinations** from train (S01,S03,S04) + validation
(S02,S05) splits. Vehicle IDs are globally consistent across all cameras --
this is a MTMC benchmark with 880 distinct vehicle identities. Only S06 (test,
no GT) is excluded.

In [ ]:
# -- Locate CityFlowV2 data --
TARGET_CAMS = set()  # empty = use all available cameras
MAX_CROPS_PER_ID_CAM = 20
MIN_AREA = 2000
MIN_BBOX_SIDE = 30
TRAIN_RATIO = 0.7
MIN_CAMS_FOR_EVAL = 2
SEED = 42

for _subpath in ["AIC22_Track1_MTMC_Tracking/train", "train"]:
    RAW_ROOT = CITYFLOW_DIR / _subpath
    if RAW_ROOT.exists():
        break

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"CityFlowV2 raw train split not found: {RAW_ROOT}")

camera_dirs = sorted(path for path in RAW_ROOT.glob("S*/c*") if path.is_dir())
if TARGET_CAMS:
    camera_dirs = [p for p in camera_dirs if f"{p.parent.name}_{p.name}" in TARGET_CAMS]

CAMERAS = [f"{path.parent.name}_{path.name}" for path in camera_dirs]
cam_dir_map = {f"{path.parent.name}_{path.name}": path for path in camera_dirs}

if not CAMERAS:
    raise RuntimeError(f"No cameras found under {RAW_ROOT}")

print(f"CityFlowV2 raw root: {RAW_ROOT}")
print(f"Using {len(CAMERAS)} cameras: {CAMERAS}")

In [ ]:
# -- Extract vehicle crops from GT + video --
# Crops go to /tmp/ to preserve /kaggle/working/ space
import gc

CROP_DIR = Path("/tmp/cityflowv2_crops")
CROP_DIR.mkdir(parents=True, exist_ok=True)


def load_gt(gt_path):
    rows = []
    with open(gt_path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, tid = int(parts[0]), int(parts[1])
            x, y, w, h = int(float(parts[2])), int(float(parts[3])), int(float(parts[4])), int(float(parts[5]))
            rows.append((frame, tid, x, y, w, h))
    return rows


def extract_crops_from_camera(cam_name, cam_dir, crop_dir, max_crops, min_area):
    gt_file = cam_dir / "gt" / "gt.txt"
    video_file = cam_dir / "vdo.avi"
    if not gt_file.exists() or not video_file.exists():
        print(f"  SKIP {cam_name}: missing GT or video")
        return {}

    gt = load_gt(str(gt_file))
    if not gt:
        print(f"  {cam_name}: empty GT file")
        return {}

    id_dets = defaultdict(list)
    for frame, tid, x, y, w, h in gt:
        if w * h < min_area or w < MIN_BBOX_SIDE or h < MIN_BBOX_SIDE:
            continue
        id_dets[tid].append((frame, x, y, w, h))

    samples = {}
    for tid, dets in id_dets.items():
        dets.sort(key=lambda item: item[0])
        if len(dets) <= max_crops:
            sampled = dets
        else:
            sampled_indices = np.linspace(0, len(dets) - 1, num=max_crops, dtype=int)
            sampled = [dets[index] for index in sampled_indices]
        samples[tid] = sampled

    dets_by_frame = defaultdict(list)
    for tid, sampled_dets in samples.items():
        for frame_id, x, y, w, h in sampled_dets:
            dets_by_frame[frame_id].append((tid, x, y, w, h))

    if not dets_by_frame:
        print(f"  {cam_name}: no valid detections after filtering")
        return {}

    crops = defaultdict(list)
    cap = cv2.VideoCapture(str(video_file))
    if not cap.isOpened():
        print(f"  SKIP {cam_name}: failed to open video")
        return {}

    crop_count = 0
    for frame_id in sorted(dets_by_frame.keys()):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id - 1)
        ok, frame = cap.read()
        if not ok:
            continue
        frame_h, frame_w = frame.shape[:2]
        for tid, x, y, w, h in dets_by_frame[frame_id]:
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(frame_w, x + w), min(frame_h, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            fname = f"{tid:04d}_{cam_name}_f{frame_id:06d}.jpg"
            fpath = crop_dir / fname
            if not cv2.imwrite(str(fpath), crop):
                continue
            crops[tid].append(str(fpath))
            crop_count += 1

    cap.release()
    gc.collect()
    print(f"  {cam_name}: {crop_count} crops from {len(crops)} vehicles")
    return dict(crops)


existing_crops = list(CROP_DIR.glob("*.jpg"))
if len(existing_crops) > 100:
    print(f"Reusing {len(existing_crops)} existing crops from {CROP_DIR}")
    all_crops = defaultdict(lambda: defaultdict(list))
    for fpath in sorted(existing_crops):
        parts = fpath.stem.split("_")
        tid = int(parts[0])
        cam = parts[1] + "_" + parts[2]
        all_crops[tid][cam].append(str(fpath))
    all_crops = {k: dict(v) for k, v in all_crops.items()}
else:
    print("Extracting crops from videos...")
    if CROP_DIR.exists():
        for existing_file in CROP_DIR.glob("*.jpg"):
            existing_file.unlink()
    all_crops = defaultdict(lambda: defaultdict(list))
    for cam in CAMERAS:
        cam_crops = extract_crops_from_camera(cam, cam_dir_map[cam], CROP_DIR, MAX_CROPS_PER_ID_CAM, MIN_AREA)
        for tid, paths in cam_crops.items():
            all_crops[tid][cam].extend(paths)
    all_crops = {k: dict(v) for k, v in all_crops.items()}

total = sum(sum(len(v) for v in cams.values()) for cams in all_crops.values())
n_ids = len(all_crops)
multi_cam = sum(1 for cams in all_crops.values() if len(cams) >= MIN_CAMS_FOR_EVAL)
print(f"\nTotal: {total} crops, {n_ids} vehicle IDs ({multi_cam} with >={MIN_CAMS_FOR_EVAL} cameras)")

scene_ids = defaultdict(set)
for tid, cams in all_crops.items():
    for cam_name in cams:
        scene = cam_name.split("_")[0]
        scene_ids[scene].add(tid)
cross_scene = 0
for tid, cams in all_crops.items():
    scenes = {c.split("_")[0] for c in cams}
    if len(scenes) > 1:
        cross_scene += 1
print(f"\nID sanity check:")
for s in sorted(scene_ids):
    print(f"  {s}: {len(scene_ids[s])} vehicle IDs")
print(f"  Cross-scene vehicles (same ID in 2+ scenes): {cross_scene}")
if n_ids > 880:
    print(f"  WARNING: {n_ids} IDs found but CityFlowV2 has 880 annotated. Possible ID collision!")
else:
    print(f"  OK: {n_ids} IDs <= 880 official annotated identities")

In [ ]:
# -- Build train / query / gallery splits --
if not all_crops:
    raise RuntimeError(
        f"No crops extracted! CAMERAS={CAMERAS}, CROP_DIR={CROP_DIR}\n"
        "The download or extraction likely failed. Check the download cell output."
    )

rng = np.random.RandomState(SEED)

multi_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) >= MIN_CAMS_FOR_EVAL)
single_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) < MIN_CAMS_FOR_EVAL)

# If no multi-cam IDs, relax to single-cam training (evaluation won't be cross-camera)
if not multi_cam_ids:
    print(f"WARNING: No vehicles seen in >={MIN_CAMS_FOR_EVAL} cameras.")
    print(f"  Using single-cam IDs for training (no cross-camera eval possible).")
    all_ids = sorted(all_crops.keys())
    rng.shuffle(all_ids)
    n_train = max(int(len(all_ids) * TRAIN_RATIO), 1)
    multi_cam_ids = all_ids  # treat all as "multi" for splitting
    single_cam_ids = []

rng.shuffle(multi_cam_ids)
n_train = int(len(multi_cam_ids) * TRAIN_RATIO)
train_ids = set(multi_cam_ids[:n_train])
eval_ids = set(multi_cam_ids[n_train:])

cam_names = sorted({cam for cams in all_crops.values() for cam in cams})
cam2id = {c: i for i, c in enumerate(cam_names)}

train_data, query_data, gallery_data = [], [], []

train_pid_set = sorted(train_ids)
pid2label = {tid: i for i, tid in enumerate(train_pid_set)}

for tid in sorted(train_ids):
    label = pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for p in paths:
            train_data.append((p, label, camid))

eval_pid2label = {tid: i for i, tid in enumerate(sorted(eval_ids))}
for tid in sorted(eval_ids):
    pid = eval_pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        if not paths:
            continue
        camid = cam2id[cam_name]
        idx = rng.randint(0, len(paths))
        query_data.append((paths[idx], pid, camid))
        for i, p in enumerate(paths):
            if i != idx:
                gallery_data.append((p, pid, camid))

distractor_pid = len(eval_ids)
for tid in sorted(single_cam_ids):
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for p in paths:
            gallery_data.append((p, distractor_pid, camid))
    distractor_pid += 1

num_classes = max(len(set(pid for _, pid, _ in train_data)), 1)
num_cameras = max(len(cam_names), 1)

CAN_EVAL = len(query_data) > 0 and len(gallery_data) > 0
print(f"Train:   {len(train_data)} images, {num_classes} IDs, {num_cameras} cameras")
print(f"Query:   {len(query_data)} images, {len(eval_ids)} IDs")
print(f"Gallery: {len(gallery_data)} images")
print(f"Cameras: {cam_names}")
print(f"Evaluation possible: {CAN_EVAL}")

## 3. Data Loading + Losses

In [ ]:
# -- CLIP normalization (must match VeRi-776 pretrained model) --
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]
H, W = 384, 384  # 384px retrain of the proven NB09 recipe

train_tf = T.Compose([
    T.Resize((H + 16, W + 16), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((H, W)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    T.RandomErasing(p=0.5, value="random"),
])

test_tf = T.Compose([
    T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])


class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        path, pid, cam = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, pid, cam, path


class PKSampler(Sampler):
    def __init__(self, data_source, p=16, k=4):
        self.p, self.k = p, k
        self.pid_to_idx = defaultdict(list)
        for i, (_, pid, _) in enumerate(data_source):
            self.pid_to_idx[pid].append(i)
        self.pids = list(self.pid_to_idx.keys())
        self.batch_size = p * k
    def __iter__(self):
        np.random.shuffle(self.pids)
        batch = []
        for pid in self.pids:
            idxs = self.pid_to_idx[pid]
            sel = np.random.choice(idxs, self.k, replace=len(idxs) < self.k).tolist()
            batch.extend(sel)
            if len(batch) >= self.batch_size:
                yield from batch[:self.batch_size]
                batch = batch[self.batch_size:]
        if batch:
            yield from batch
    def __len__(self):
        return len(self.pids) * self.k


BATCH = 14 * NUM_GPUS  # 384px uses substantially more memory per image
P_IDS = 7 * NUM_GPUS   # paired with 2 instances per ID to fit the 384px batch budget

train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

sampler = PKSampler(train_data, p=P_IDS, k=2)
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          num_workers=4, pin_memory=True, drop_last=True)
query_loader = DataLoader(query_ds, batch_size=64, num_workers=4, pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=64, num_workers=4, pin_memory=True)

print(f"Train: {len(train_loader)} batches (batch={BATCH})")
print(f"Query: {len(query_loader)} | Gallery: {len(gallery_loader)}")

In [ ]:
# -- Losses (fp32-stable, proven in NB08) --
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs.float(), dim=1)
        with torch.no_grad():
            oh = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
            smooth = (1 - self.epsilon) * oh + self.epsilon / self.num_classes
        loss = (-smooth * log_probs).sum(dim=1).mean()
        return loss


class TripletLossHardMining(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, pids):
        feats = F.normalize(feats.float(), p=2, dim=1)
        n = feats.size(0)
        dist = torch.cdist(feats, feats, p=2)
        mask_pos = pids.unsqueeze(0).eq(pids.unsqueeze(1))
        mask_neg = ~mask_pos
        dist_pos = dist.clone(); dist_pos[~mask_pos] = 0
        hardest_pos, _ = dist_pos.max(dim=1)
        dist_neg = dist.clone(); dist_neg[~mask_neg] = float('inf')
        hardest_neg, _ = dist_neg.min(dim=1)
        y = torch.ones(n, device=feats.device)
        return self.ranking_loss(hardest_neg, hardest_pos, y)


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, feats, labels):
        feats = feats.float()
        centers_batch = self.centers[labels]
        diff = feats - centers_batch
        loss = (diff * diff).sum(1).mean()
        return loss


print("Losses: CE+LS(eps=0.05), TripletLoss(m=0.3), CenterLoss")

## 4. Evaluation Functions

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device="cuda", flip=True, pass_cams=False):
    model.eval()
    feats, pids, cams = [], [], []
    for imgs, pid, cam, _ in loader:
        imgs = imgs.to(device)
        kwargs = {"cam_ids": cam.to(device).long()} if pass_cams else {}
        f = model(imgs, **kwargs)
        if isinstance(f, (tuple, list)):
            f = f[-1]
        if flip:
            ff = model(torch.flip(imgs, [3]), **kwargs)
            if isinstance(ff, (tuple, list)):
                ff = ff[-1]
            f = (f + ff) / 2
        f = F.normalize(f, p=2, dim=1)
        feats.append(f.cpu().numpy())
        pids.append(pid.numpy())
        cams.append(cam.numpy())
    if not feats:
        return np.zeros((0, 768)), np.zeros(0, dtype=int), np.zeros(0, dtype=int)
    return np.concatenate(feats), np.concatenate(pids), np.concatenate(cams)


def eval_market1501(distmat, q_pids, g_pids, q_cams, g_cams, max_rank=50):
    if distmat.shape[0] == 0 or distmat.shape[1] == 0:
        return 0.0, np.zeros(max_rank)
    nq = distmat.shape[0]
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, None]).astype(np.int32)
    all_cmc, all_AP = [], []
    for qi in range(nq):
        order = indices[qi]
        remove = (g_pids[order] == q_pids[qi]) & (g_cams[order] == q_cams[qi])
        raw_cmc = matches[qi][~remove]
        if raw_cmc.sum() == 0:
            continue
        cmc = raw_cmc.cumsum(); cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        n_rel = raw_cmc.sum()
        tmp = raw_cmc.cumsum()
        prec = tmp / (np.arange(len(tmp)) + 1.0)
        all_AP.append((prec * raw_cmc).sum() / n_rel)
    if not all_AP:
        return 0.0, np.zeros(max_rank)
    return float(np.mean(all_AP)), np.array(all_cmc).mean(0)


def compute_reranking(qf, gf, k1=20, k2=6, lam=0.3):
    if qf.shape[0] == 0 or gf.shape[0] == 0:
        return np.zeros((qf.shape[0], gf.shape[0]))
    all_f = np.concatenate([qf, gf], axis=0)
    N, nq = all_f.shape[0], qf.shape[0]
    od = np.clip(2.0 - 2.0 * (all_f @ all_f.T), 0, None)
    ir = np.argsort(od, axis=1)
    V = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        fwd = ir[i, :k1+1]
        kr = [c for c in fwd if i in ir[c, :k1+1]]
        kr = np.array(kr); kr_exp = kr.copy()
        for c in kr:
            cf = ir[c, :int(round(k1/2))+1]
            ckr = [cc for cc in cf if c in ir[cc, :int(round(k1/2))+1]]
            if len(ckr) > 2/3 * len(cf):
                kr_exp = np.union1d(kr_exp, ckr)
        w = np.exp(-od[i, kr_exp])
        V[i, kr_exp] = w / (w.sum() + 1e-12)
    if k2 > 0:
        Vqe = np.zeros_like(V)
        for i in range(N):
            Vqe[i] = V[ir[i, :k2+1]].mean(0)
        V = Vqe
    jac = np.zeros((nq, N-nq), dtype=np.float32)
    for i in range(nq):
        mn = np.minimum(V[i], V[nq:]); mx = np.maximum(V[i], V[nq:])
        jac[i] = 1 - mn.sum(1) / (mx.sum(1) + 1e-12)
    return jac * (1 - lam) + od[:nq, nq:] * lam


def full_eval(model, ql, gl, device="cuda", rerank=True, pass_cams=False):
    qf, qp, qc = extract_features(model, ql, device, pass_cams=pass_cams)
    gf, gp, gc = extract_features(model, gl, device, pass_cams=pass_cams)
    if qf.shape[0] == 0 or gf.shape[0] == 0:
        print("  [WARN] Empty query or gallery -- skipping eval")
        return 0.0, np.zeros(50), None, None
    dist = 1.0 - qf @ gf.T
    mAP, cmc = eval_market1501(dist, qp, gp, qc, gc)
    mAP_rr, cmc_rr = None, None
    if rerank:
        dist_rr = compute_reranking(qf, gf)
        mAP_rr, cmc_rr = eval_market1501(dist_rr, qp, gp, qc, gc)
    return mAP, cmc, mAP_rr, cmc_rr

print("Evaluation ready")

## 5. TransReID Model

Same architecture as NB08 (v15): ViT-Base/16 CLIP + SIE + JPM + BNNeck routing fix.

**Key difference**: We initialize from VeRi-776 pretrained weights instead of
raw CLIP, then fine-tune on CityFlowV2. This gives us:
- Vehicle-specific feature representations from VeRi-776
- Domain adaptation to real multi-camera traffic footage
- Camera-aware SIE re-learned for CityFlowV2 camera topology

In [ ]:
import timm

def find_clip_vit_base():
    try:
        tags = timm.list_pretrained('vit_base_patch16_clip*224*')
    except Exception:
        tags = []
    for t in sorted(tags):
        if 'openai' in t and 'ft' not in t:
            return t
    for t in sorted(tags):
        if 'openai' in t:
            return t
    if tags:
        return sorted(tags)[0]
    clip_models = timm.list_models("*vit_base*patch16*clip*224*")
    if clip_models:
        return sorted(clip_models)[0]
    raise RuntimeError("No CLIP ViT-Base found in timm!")

VIT_MODEL = find_clip_vit_base()
print(f"Backbone: {VIT_MODEL}")


class TransReID(nn.Module):
    def __init__(self, num_classes, num_cameras=0, embed_dim=768,
                 vit_model="vit_base_patch16_clip_224", pretrained=True,
                 sie_camera=True, jpm=True, img_size=224):
        super().__init__()
        self.sie_camera = sie_camera and num_cameras > 0
        self.jpm = jpm
        self.vit = timm.create_model(vit_model, pretrained=pretrained,
                                     num_classes=0, img_size=img_size)
        self.vit_dim = self.vit.embed_dim

        self.num_blocks = len(self.vit.blocks)

        if self.sie_camera:
            self.sie_embed = nn.Parameter(torch.zeros(num_cameras, 1, self.vit_dim))
            nn.init.trunc_normal_(self.sie_embed, std=0.02)

        self.bn = nn.BatchNorm1d(self.vit_dim)
        self.bn.bias.requires_grad_(False)

        self.proj = nn.Linear(self.vit_dim, embed_dim, bias=False) if embed_dim != self.vit_dim else nn.Identity()
        self.cls_head = nn.Linear(embed_dim, num_classes, bias=False)
        if isinstance(self.proj, nn.Linear):
            nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
        nn.init.normal_(self.cls_head.weight, std=0.001)

        if self.jpm:
            self.bn_jpm = nn.BatchNorm1d(self.vit_dim)
            self.bn_jpm.bias.requires_grad_(False)
            self.jpm_cls = nn.Linear(self.vit_dim, num_classes, bias=False)
            nn.init.normal_(self.jpm_cls.weight, std=0.001)

        print(f"TransReID: {vit_model}, dim={self.vit_dim}, embed={embed_dim}, "
              f"SIE={self.sie_camera}({num_cameras}), JPM={jpm}, img={img_size}")

    def forward(self, x, cam_ids=None):
        B = x.shape[0]
        x = self.vit.patch_embed(x)
        if hasattr(self.vit, '_pos_embed'):
            x = self.vit._pos_embed(x)
        else:
            cls_tok = self.vit.cls_token.expand(B, -1, -1)
            x = torch.cat([cls_tok, x], dim=1) + self.vit.pos_embed
            if hasattr(self.vit, 'pos_drop'):
                x = self.vit.pos_drop(x)

        if self.sie_camera and cam_ids is not None:
            x = x + self.sie_embed[cam_ids]

        if hasattr(self.vit, 'patch_drop'):
            x = self.vit.patch_drop(x)
        if hasattr(self.vit, 'norm_pre'):
            x = self.vit.norm_pre(x)

        for blk in self.vit.blocks:
            x = blk(x)
        x = self.vit.norm(x)

        g = x[:, 0]  # pre-BN: for triplet + center
        bn = self.bn(g)  # BNNeck: for CE + inference
        proj = self.proj(bn)

        if self.training:
            cls = self.cls_head(proj)
            if self.jpm:
                patches = x[:, 1:]
                idx = torch.randperm(patches.size(1), device=x.device)
                s = patches[:, idx]
                mid = s.size(1) // 2
                jf = (s[:, :mid].mean(1) + s[:, mid:].mean(1)) / 2
                return cls, g, self.jpm_cls(self.bn_jpm(jf))
            return cls, g
        return F.normalize(proj, p=2, dim=1)

    def get_llrd_param_groups(self, backbone_lr, head_lr, decay=0.75):
        groups = {}
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            if name.startswith('vit.'):
                if 'blocks.' in name:
                    block_idx = int(name.split('blocks.')[1].split('.')[0])
                    depth = block_idx + 1
                elif any(k in name for k in ['patch_embed', 'cls_token', 'pos_embed', 'norm_pre']):
                    depth = 0
                else:
                    depth = self.num_blocks + 1
                scale = decay ** (self.num_blocks + 1 - depth)
                lr = backbone_lr * scale
                gk = f"bb_d{depth}"
            else:
                lr = head_lr
                gk = "head"
            if gk not in groups:
                groups[gk] = {"params": [], "lr": lr}
            groups[gk]["params"].append(param)
        return sorted(groups.values(), key=lambda x: x["lr"])


model = TransReID(
    num_classes=num_classes, num_cameras=num_cameras,
    embed_dim=768, vit_model=VIT_MODEL, sie_camera=True, jpm=True,
    img_size=H,
).to(DEVICE)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# -- Load VeRi-776 pretrained weights (domain adaptation) --
# Backbone + BNNeck transfer; classifier head re-initialized for CityFlowV2.
# pos_embed is skipped because VeRi's is 224-sized; we keep timm's resized CLIP version.
PRETRAINED_PATH = None

search_paths = [
    # Kaggle Models (attached via kernel-metadata.json model_sources)
    Path("/kaggle/input/models/mrkdagods/transreid-veri/other/default/1/transreid_veri_best.pth"),
    # Fallbacks
    Path("models/reid/vehicle_transreid_vit_base_veri776.pth"),
]

for p in search_paths:
    if p.exists():
        PRETRAINED_PATH = p
        break

if PRETRAINED_PATH is not None:
    print(f"Loading VeRi-776 pretrained weights from {PRETRAINED_PATH}")
    ckpt = torch.load(str(PRETRAINED_PATH), map_location="cpu")
    state = ckpt.get("state_dict", ckpt)

    # Skip classifier head, SIE, and pos_embed (shape mismatch: 224 vs 384)
    skip_keys = ["cls_head", "jpm_cls", "sie_embed", "pos_embed"]
    filtered = {}
    skipped = []
    for k, v in state.items():
        if any(sk in k for sk in skip_keys):
            skipped.append(k)
            continue
        filtered[k] = v

    missing, unexpected = model.load_state_dict(filtered, strict=False)
    print(f"  Loaded: {len(filtered)} params")
    print(f"  Skipped (re-init): {skipped}")
    print(f"  Missing (new): {missing}")
    print(f"  Backbone + BNNeck transferred from VeRi-776")
else:
    print("No VeRi-776 pretrained weights found -- training from CLIP init")
    print("For best results, attach NB08 output as Kaggle dataset")

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"  Wrapped in DataParallel ({NUM_GPUS} GPUs)")

## 6. Training

Fine-tuning strategy (tuned for CityFlowV2 384px retraining):
- **120 epochs**, 384x384 resolution
- **Lower LR**: backbone_lr=1e-4, label smoothing=0.05 (sharper R1)
- Center loss delayed to epoch 15
- Losses: CE(eps=0.05) + Triplet(0.3) + Center(5e-4)
- Eval every 20 epochs

In [ ]:
# -- Training config (fine-tuning) --
ce_loss = CrossEntropyLabelSmooth(num_classes, 0.05).to(DEVICE)  # reduced from 0.1 for sharper R1
tri_loss = TripletLossHardMining(margin=0.3).to(DEVICE)
ctr_loss = CenterLoss(num_classes, 768).to(DEVICE)
CENTER_WEIGHT = 5e-4
CENTER_START = 15

raw_model = model.module if hasattr(model, 'module') else model

backbone_lr = 1e-4
head_lr = 1e-3
wd = 5e-4
llrd_factor = 0.75

param_groups = raw_model.get_llrd_param_groups(backbone_lr, head_lr, decay=llrd_factor)
optimizer = torch.optim.AdamW(param_groups, weight_decay=wd)
center_optimizer = torch.optim.SGD(ctr_loss.parameters(), lr=0.5)
base_lrs = [pg["lr"] for pg in optimizer.param_groups]

EPOCHS = 120
WARMUP = 10

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP)
scaler = torch.amp.GradScaler("cuda")

history = {"loss": [], "mAP": [], "R1": [], "mAP_rr": [], "R1_rr": []}
best_mAP = 0.0

print("=" * 70)
print(f"  Fine-tuning TransReID on CityFlowV2 ({num_classes} IDs, {num_cameras} cameras)")
print(f"  Init: {'VeRi-776 pretrained' if PRETRAINED_PATH else 'CLIP only'}")
print(f"  Losses: CE(eps=0.05) + Triplet(0.3) + Center(5e-4, delayed@ep{CENTER_START})")
print(f"  LR: backbone={backbone_lr}, head={head_lr}, LLRD={llrd_factor}")
print(f"  Epochs: {EPOCHS}, warmup: {WARMUP}, resolution: {H}x{W}")
print("=" * 70)

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    rl, nb = 0.0, 0
    use_center = (epoch >= CENTER_START)

    if epoch < WARMUP:
        wf = (epoch + 1) / WARMUP
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i] * wf
    elif epoch == WARMUP:
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i]

    for imgs, pids, cams, _ in train_loader:
        imgs, pids, cams = imgs.to(DEVICE), pids.to(DEVICE).long(), cams.to(DEVICE).long()
        optimizer.zero_grad()
        if use_center:
            center_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            out = model(imgs, cam_ids=cams)
            if len(out) == 3:
                c, f, jc = out
                loss = ce_loss(c, pids) + tri_loss(f, pids) + 0.5 * ce_loss(jc, pids)
            else:
                c, f = out
                loss = ce_loss(c, pids) + tri_loss(f, pids)

        if use_center:
            center_l = ctr_loss(f.float(), pids)
            total_loss = loss + CENTER_WEIGHT * center_l
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            scaler.unscale_(center_optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.step(center_optimizer)
        else:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)

        scaler.update()
        rl += loss.item() if not torch.isnan(loss) else 0.0
        nb += 1

    if epoch >= WARMUP:
        scheduler.step()
    history["loss"].append(rl / max(nb, 1))

    if (epoch + 1) % 10 == 0:
        hd_lr = optimizer.param_groups[-1]["lr"]
        ctr_tag = " [+center]" if use_center else ""
        print(f"Epoch {epoch+1:3d} | Loss {rl/max(nb,1):.4f} | head_lr={hd_lr:.2e}{ctr_tag}")

    if (epoch + 1) % 20 == 0 or epoch == EPOCHS - 1:
        mAP, cmc, mAP_rr, cmc_rr = full_eval(model, query_loader, gallery_loader,
                                               DEVICE, pass_cams=True)
        history["mAP"].append(mAP); history["R1"].append(cmc[0])
        history["mAP_rr"].append(mAP_rr or 0)
        history["R1_rr"].append(cmc_rr[0] if cmc_rr is not None else 0)
        is_best = mAP > best_mAP
        if is_best:
            best_mAP = mAP
            _state = (model.module if hasattr(model, 'module') else model).state_dict()
            torch.save(_state, OUTPUT_DIR / "transreid_cityflowv2_384px_best.pth")
        tag = " BEST" if is_best else ""
        print(f"  mAP: {mAP:.4f}, R1: {cmc[0]:.4f}")
        if mAP_rr: print(f"  mAP(RR): {mAP_rr:.4f}, R1(RR): {cmc_rr[0]:.4f}{tag}")

elapsed = time.time() - t0
print(f"\nDone in {elapsed/3600:.1f}h | Best mAP: {best_mAP:.4f}")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("TransReID (CLIP) -- CityFlowV2 Fine-Tune (384px)", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(history["loss"], "b-", alpha=0.7)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)

ax = axes[1]
ee = [20*(i+1) for i in range(len(history["mAP"]))]
if history["mAP"]:
    ax.plot(ee, [m*100 for m in history["mAP"]], "r-o", label="mAP")
    ax.plot(ee, [r*100 for r in history["R1"]], "b-s", label="R1")
    if any(history["mAP_rr"]):
        ax.plot(ee, [m*100 for m in history["mAP_rr"]], "r--^", label="mAP(RR)")
ax.set_xlabel("Epoch"); ax.set_ylabel("%"); ax.set_title("Metrics")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Export Model

In [ ]:
# -- Export best model --
best_state = torch.load(OUTPUT_DIR / "transreid_cityflowv2_384px_best.pth", map_location="cpu")
export_path = EXPORT_DIR / "transreid_cityflowv2_384px_best.pth"
torch.save({"state_dict": best_state}, export_path)
print(f"Model exported to {export_path} ({export_path.stat().st_size / 1e6:.1f} MB)")

metadata = {
    "task": "vehicle_reid",
    "dataset": "cityflowv2",
    "source_dataset": "AI City Challenge 2022 Track 1",
    "init_weights": "veri776" if PRETRAINED_PATH else "clip",
    "training": f"TransReID ViT-Base CLIP -> CityFlowV2 fine-tune ({EPOCHS}ep)",
    "model": {
        "architecture": VIT_MODEL,
        "type": "transreid",
        "tricks": ["SIE", "JPM", "BNNeck", "CE+LS(0.05)", "TripletLoss(m=0.3)",
                    f"CenterLoss(delayed@ep{CENTER_START})", "CosLR", "RE", "CLIP-norm",
                    f"LLRD({llrd_factor})"],
        "embedding_dim": 768,
        "input_size": [384, 384],
        "normalization": {"mean": CLIP_MEAN, "std": CLIP_STD},
        "num_cameras": num_cameras,
        "num_classes": num_classes,
        "best_mAP": float(best_mAP),
        "best_mAP_rr": float(history["mAP_rr"][-1]) if history["mAP_rr"] else None,
        "epochs": EPOCHS,
        "backbone_lr": backbone_lr,
        "head_lr": head_lr,
        "center_loss_weight": CENTER_WEIGHT,
        "center_loss_start_epoch": CENTER_START,
        "label_smoothing": 0.05,
    },
    "cameras": CAMERAS,
    "split": {
        "train_images": len(train_data),
        "train_ids": num_classes,
        "query_images": len(query_data),
        "gallery_images": len(gallery_data),
        "eval_ids": len(eval_ids),
    },
}

meta_path = EXPORT_DIR / "vehicle_reid_cityflowv2_384px_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to {meta_path}")

print("\n" + "=" * 60)
print("RESULTS SUMMARY -- CityFlowV2 Vehicle ReID")
print("=" * 60)
print(f"Dataset:    CityFlowV2 ({len(CAMERAS)} cameras)")
print(f"Train:      {len(train_data)} images, {num_classes} IDs")
print(f"Eval:       {len(query_data)} query + {len(gallery_data)} gallery")
print(f"Model:      TransReID ViT-Base/16 CLIP")
print(f"Init:       {'VeRi-776 pretrained' if PRETRAINED_PATH else 'CLIP only'}")
print(f"Epochs:     {EPOCHS}")
if history["mAP"]:
    print(f"Best mAP:   {best_mAP*100:.2f}%")
    print(f"Best R1:    {max(history['R1'])*100:.2f}%")
    if history["mAP_rr"] and any(history["mAP_rr"]):
        print(f"Best mAP(RR): {max(history['mAP_rr'])*100:.2f}%")
print("=" * 60)

## 9. Inference Integration

### Using the trained model in the MTMC pipeline

After training, download the exported model and metadata:

```bash
# Copy exported files to local project
cp transreid_cityflowv2_384px_best.pth  models/reid/
cp vehicle_reid_cityflowv2_384px_metadata.json       models/reid/
```

### Update config to use CityFlowV2-trained model

In `configs/default.yaml` or `configs/datasets/cityflowv2.yaml`:

```yaml
stage2:
  reid:
    vehicle:
      model_name: "transreid"
      weights_path: "models/reid/transreid_cityflowv2_384px_best.pth"
      embedding_dim: 768
      input_size: [256, 256]
      vit_model: "vit_base_patch16_clip_224.openai"
      num_cameras: 6
      clip_normalization: true
```

### Run full pipeline with MTMC evaluation

```bash
# Run full pipeline on CityFlowV2
python scripts/run_pipeline.py --config configs/datasets/cityflowv2.yaml

# Or evaluate ReID only
python scripts/eval_cityflowv2_reid.py \
    --weights models/reid/transreid_cityflowv2_384px_best.pth
```

### Expected metrics
| Metric | ReID (Stage 2) | MTMC (Stage 5) |
|--------|---------------|----------------|
| mAP | 40-65% | -- |
| Rank-1 | 55-80% | -- |
| IDF1 | -- | 70-84% |
| HOTA | -- | 50-65% |
| MOTA | -- | 60-78% |